# Novo treino do BERTimbau de referência (semente 42) e gravação dos pesos no Google Drive

Versão reduzida do `pipeline_uan_colab_v10.ipynb`. Faz **apenas** o necessário para recuperar o modelo usado no sistema:

1. constrói o corpus híbrido v13 e a divisão treino/validação/teste **com os mesmos comandos** da corrida de referência;
2. **confirma** que a validação e o teste são exactamente os da corrida de referência (impressão digital das etiquetas);
3. treina o BERTimbau com a configuração de referência (semente 42, 3 épocas, lote 16, taxa 2×10⁻⁵, 256 subpalavras, entropia cruzada ponderada);
4. compara o resultado com o da corrida de referência (F1-macro 0,807);
5. grava o modelo no Google Drive, no formato que o sistema `sistema_uan` carrega.

Não treina a Floresta Aleatória, o SVM nem o Bi-LSTM, nem repete as secções 5 a 9: esses resultados já estão no Capítulo 4.

**Antes de começar:** `Ambiente de execução → Alterar tipo de ambiente de execução → T4 GPU`. O treino demora cerca de 40 a 60 minutos numa T4. Mantenha o separador aberto.

## 0. Verificar a GPU e montar o Google Drive

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), "Sem GPU. Escolha 'T4 GPU' em Ambiente de execução → Alterar tipo de ambiente de execução."
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
PASTA_DRIVE = "/content/drive/MyDrive/dissertacao_uan"                  # tudo o que tem de sobreviver fica aqui
DESTINO_MODELO = f"{PASTA_DRIVE}/bertimbau_uan_v13_s42"                  # pasta que o sistema usa (UAN_MODELO)
os.makedirs(PASTA_DRIVE, exist_ok=True)
print("Pasta no Drive:", PASTA_DRIVE)

## 1. Código e dados (`uan_pipeline.zip`)

Se o zip já estiver em `MyDrive/dissertacao_uan/uan_pipeline.zip`, é usado directamente; senão, pede o carregamento e guarda uma cópia no Drive para a próxima vez. O zip já inclui `data/corpus_ground_truth_com_texto_v13.csv`.

In [ ]:
import shutil, zipfile
ZIP_DRIVE = f"{PASTA_DRIVE}/uan_pipeline.zip"
os.chdir("/content")
if os.path.exists(ZIP_DRIVE):
    shutil.copy(ZIP_DRIVE, "uan_pipeline.zip"); print("Zip lido do Drive.")
else:
    from google.colab import files
    up = files.upload()                                   # escolha uan_pipeline.zip
    nome = [f for f in up if f.endswith(".zip")][0]
    if nome != "uan_pipeline.zip":
        os.replace(nome, "uan_pipeline.zip")
    shutil.copy("uan_pipeline.zip", ZIP_DRIVE); print("Cópia guardada no Drive.")

with zipfile.ZipFile("uan_pipeline.zip") as z:
    z.extractall("/content")
os.chdir("/content/uan_pipeline")
assert os.path.exists("data/corpus_ground_truth_com_texto_v13.csv"), "O zip não traz o corpus v13."
print("Directório:", os.getcwd())

In [ ]:
# 'datasets<4.0.0' tem de ser instalado antes do primeiro import de datasets nesta sessão
!pip install -q -r requirements.txt

## 2. Corpus híbrido v13 (mesmo comando da corrida de referência)

In [ ]:
!python src/build_public_corpus.py \
    --pilot-angolano data/corpus_ground_truth_com_texto_v13.csv \
    --out data/corpus_hibrido_final.csv \
    --report data/harmonizacao_relatorio.json

In [ ]:
import json
rel = json.load(open("data/harmonizacao_relatorio.json", encoding="utf-8"))
ESPERADO = {"total_apos_dedup": 26859, "por_fonte": {"ToLD-BR": 12709, "Fake.br": 7199, "HateBR": 4378, "Piloto_Angola": 2573}}
print("Total:", rel["total_apos_dedup"], "| por fonte:", rel["por_fonte"])
if rel["total_apos_dedup"] != ESPERADO["total_apos_dedup"] or rel["por_fonte"] != ESPERADO["por_fonte"]:
    print("\n⚠️ O corpus difere do da corrida de referência (algum conjunto público mudou no Hugging Face?).")
    print("   Pode continuar, mas o resultado deixa de ser directamente comparável com o Capítulo 4.")
else:
    print("\n✅ Corpus idêntico ao da corrida de referência.")

## 3. Divisão treino/validação/teste e verificação de que é a mesma

In [ ]:
!python src/split_and_balance.py --in data/corpus_hibrido_final.csv \
    --out-dir data/splits --strat-col categoria_final \
    --train-only-origem enriquecido_lexico enriquecido_video_tema

In [ ]:
import hashlib, pandas as pd
LABELS = ["Normal", "Desinformacao", "Odio", "Mobilizacao"]
REFERENCIA = {   # impressão digital das etiquetas da corrida de referência (val_predictions / test_predictions)
    "val":  (3986, [2843, 580, 516, 47], "30c53837f500973a123fed60cbb379de"),
    "test": (3987, [2843, 580, 517, 47], "a827ac39358164b48589291ade21bf33"),
}
IGUAL = True
for s, (n, cont, h) in REFERENCIA.items():
    y = pd.read_csv(f"data/splits/{s}.csv")["categoria_final"].map({l: i for i, l in enumerate(LABELS)}).astype(int).tolist()
    h2 = hashlib.md5(",".join(map(str, y)).encode()).hexdigest()
    c2 = [y.count(i) for i in range(4)]
    ok = (len(y), c2, h2) == (n, cont, h)
    IGUAL &= ok
    print(f"{s}: n={len(y)} contagens={c2} {'✅ igual' if ok else '⚠️ DIFERENTE'} à referência")
print("\nTreino:", len(pd.read_csv("data/splits/train.csv")), "itens (esperado: 18886)")
if not IGUAL:
    print("\n⚠️ A validação/teste não coincidem com os da corrida de referência. O modelo serve para o sistema,"
          " mas os números deste treino NÃO substituem os do Capítulo 4.")

## 4. Treino do BERTimbau (configuração de referência)

Mesmo comando da secção 4.4 do notebook v10: semente 42, 3 épocas e os valores por omissão do `train_bertimbau.py` (lote 16, taxa de aprendizagem 2×10⁻⁵, comprimento máximo 256, entropia cruzada ponderada). No fim do treino, o script recarrega a época com melhor F1-macro de validação.

O treino escreve em `/content` (disco local, rápido); só o modelo final é copiado para o Drive no passo 6.

In [ ]:
!python src/train_bertimbau.py --train data/splits/train.csv \
    --val data/splits/val.csv --test data/splits/test.csv \
    --out-dir models/bertimbau_ameacas --epochs 3 --seed 42

## 5. Comparar com a corrida de referência

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score
m = json.load(open("models/bertimbau_ameacas/test_metrics.json", encoding="utf-8"))
p = np.load("models/bertimbau_ameacas/test_predictions.npz")
f1, k = m["f1_macro"], m["kappa_vs_ground_truth"]
f1_cls = f1_score(p["y_true"], p["y_pred"], average=None, labels=[0, 1, 2, 3])
REF = {"f1": 0.8067, "kappa": 0.8321, "f1_cls": [0.952, 0.947, 0.778, 0.549], "sementes": (0.7846, 0.8067)}
print(f"F1-macro: {f1:.4f}  (referência {REF['f1']:.4f}; diferença {100*(f1-REF['f1']):+.1f} p.p.)")
print(f"Kappa:    {k:.4f}  (referência {REF['kappa']:.4f})")
for l, a, b in zip(LABELS, f1_cls, REF["f1_cls"]):
    print(f"  F1 {l:<14} {a:.3f}  (referência {b:.3f})")
lo, hi = REF["sementes"]
if lo - 0.01 <= f1 <= hi + 0.01:
    print("\n✅ Dentro da variação já observada entre sementes (0,785–0,807). Pode usar este modelo no sistema.")
else:
    print("\n⚠️ Fora da variação observada entre sementes. Não use no sistema antes de rever: envie o test_metrics.json.")
print("\n" + m["classification_report"])

## 6. Gravar o modelo no Google Drive

Copia apenas o que o sistema precisa (configuração, pesos e tokenizador), acrescenta os nomes das categorias à configuração e grava as métricas e uma nota de proveniência. Os *checkpoints* intermédios (~1,3 GB cada) não são copiados.

In [ ]:
import glob, datetime
from transformers import AutoConfig
ORIGEM = "models/bertimbau_ameacas"

cfg = AutoConfig.from_pretrained(ORIGEM)
cfg.id2label = dict(enumerate(LABELS)); cfg.label2id = {l: i for i, l in enumerate(LABELS)}
cfg.save_pretrained(ORIGEM)                                     # o sistema passa a ler os nomes das categorias

if os.path.exists(DESTINO_MODELO):
    shutil.move(DESTINO_MODELO, DESTINO_MODELO + "_anterior_" + datetime.datetime.now().strftime("%Y%m%d_%H%M"))
os.makedirs(DESTINO_MODELO)
for f in os.listdir(ORIGEM):
    caminho = os.path.join(ORIGEM, f)
    if os.path.isfile(caminho) and not f.startswith("training_args"):
        shutil.copy(caminho, DESTINO_MODELO)

nota = {
    "modelo": "BERTimbau (neuralmind/bert-base-portuguese-cased) afinado para 4 categorias",
    "categorias": LABELS, "semente": 42, "epocas": 3, "lote": 16, "taxa_aprendizagem": 2e-5,
    "max_length": 256, "perda": m["loss_type"], "corpus": "híbrido v13 (26 859 registos)",
    "divisao_igual_a_referencia": bool(IGUAL), "f1_macro_teste": round(f1, 4), "kappa_teste": round(k, 4),
    "treinado_em": datetime.datetime.now().isoformat(timespec="minutes"),
    "uso": "set UAN_MODELO=<esta pasta> e set UAN_MAX_LENGTH=256 no sistema_uan",
}
json.dump(nota, open(f"{DESTINO_MODELO}/PROVENIENCIA.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

total = sum(os.path.getsize(x) for x in glob.glob(f"{DESTINO_MODELO}/*")) / 1e6
print(f"Guardado em {DESTINO_MODELO} ({total:.0f} MB):")
for f in sorted(os.listdir(DESTINO_MODELO)):
    print("  ", f)
assert os.path.exists(f"{DESTINO_MODELO}/config.json") and glob.glob(f"{DESTINO_MODELO}/*.safetensors") + glob.glob(f"{DESTINO_MODELO}/pytorch_model.bin")

## 7. Teste rápido: carregar a pasta do Drive como o sistema a carrega

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tok = AutoTokenizer.from_pretrained(DESTINO_MODELO)
mod = AutoModelForSequenceClassification.from_pretrained(DESTINO_MODELO).eval().to("cuda")
print("id2label:", mod.config.id2label)
exemplos = ["Bom dia a todos, hoje há jogo dos Palancas.",
            "Partilha antes que apaguem: estão a esconder os números verdadeiros."]
with torch.no_grad():
    enc = tok(exemplos, truncation=True, max_length=256, padding=True, return_tensors="pt").to("cuda")
    pr = torch.softmax(mod(**enc).logits, -1).cpu().numpy()
for t, p_ in zip(exemplos, pr):
    print(f"{mod.config.id2label[int(p_.argmax())]:<14} {p_.max():.2f}  {t}")
print("\n✅ O modelo carrega a partir do Drive. No PC: descarregue a pasta bertimbau_uan_v13_s42 e use-a em UAN_MODELO.")

## 8. (Opcional) Descarregar só as métricas

In [ ]:
with zipfile.ZipFile("retreino_s42_metricas.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ["test_metrics.json", "test_predictions.npz", "val_predictions.npz"]:
        z.write(f"models/bertimbau_ameacas/{f}", f)
    z.write(f"{DESTINO_MODELO}/PROVENIENCIA.json", "PROVENIENCIA.json")
    z.write("data/harmonizacao_relatorio.json", "harmonizacao_relatorio.json")
from google.colab import files
files.download("retreino_s42_metricas.zip")

## Notas

- **Reprodutibilidade:** mesmo com a mesma semente, o treino em GPU não é totalmente determinístico. Espera-se um F1-macro próximo de 0,807, dentro da variação observada entre sementes (0,785–0,807). Os números do Capítulo 4 continuam a ser os da corrida original; na dissertação basta indicar que o modelo do protótipo é um novo treino com a mesma configuração, com o F1 que sair no passo 5.
- **Comprimento máximo:** o modelo foi treinado com 256 subpalavras. No sistema, use `set UAN_MAX_LENGTH=256`.
- **Se a sessão cair a meio do treino:** volte a correr a partir do passo 0. O zip fica no Drive e não é preciso carregá-lo de novo.